In [32]:
import leafmap.leafmap as leafmap

m = leafmap.Map()
m.add_basemap("OpenTopoMap")

# Test with the local file on your hard drive
m.add_raster("data/SP_CNP_AllYears_pooled_2016-2026_COG.tif", layer_name="Local TIF test",     colormap="viridis", vmin=0, vmax=75)
m

Map(center=[57.0733245, -3.6461315], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title…

In [24]:
import seaborn as sns          # registers "mako", "rocket", "flare", "crest" with matplotlib
import leafmap.leafmap as leafmap

m = leafmap.Map()
m.add_basemap("OpenTopoMap")
m.add_raster(
    "data/SP_CNP_AllYears_pooled_2016-2026_COG.tif",
    layer_name="Local TIF test",
    colormap="mako",
    vmin=0,
    vmax=100,          # set these to your data's actual range
)
m.add_colormap(cmap="mako", vmin=0, vmax=100, label="Density")
m

ModuleNotFoundError: No module named 'seaborn'

In [28]:
# ```{python}
# #| echo: false
# #| warning: false

import json
import matplotlib as mpl
import seaborn as sns
import leafmap.foliumap as leafmap

# ── EDIT THESE ────────────────────────────────────────────────
COG_URL = "https://huggingface.co/datasets/leamhowe/cairngorms-S2-snow-persistence/blob/main/SP_CNP_AllYears_pooled_2016-2026_COG.tif"
VMIN, VMAX = 0, 100          # your data's actual value range
NODATA = 0                   # or None if the COG has no nodata value
# CENTER, ZOOM = [-33.9, 151.2], 8
# ──────────────────────────────────────────────────────────────

# Build a titiler-compatible mako colormap: {0..255: [r,g,b,a]}
mako_cmap = {
    i: [int(255 * v) for v in mpl.colors.to_rgba(c)]
    for i, c in enumerate(sns.color_palette("mako", 256).as_hex())
}
mako_hex = sns.color_palette("mako", 256).as_hex()

m = leafmap.Map(center=(57, -2.5), zoom=8)
m.add_basemap("OpenTopoMap")
m.add_cog_layer(
    COG_URL,
    name="Pooled 2016-2026",
    colormap=json.dumps(mako_cmap),
    rescale=f"{VMIN},{VMAX}",
    **({"nodata": NODATA} if NODATA is not None else {}),
)
m.add_colorbar(colors=mako_hex, vmin=VMIN, vmax=VMAX, caption="Density")
m
# ```

In [36]:
import leafmap.foliumap as leafmap

# 1. Initialize the folium-backed map
m = leafmap.Map(center=(57, -2.5), zoom=8)

VMIN, VMAX = 0, 100       # your data's actual value range


# 2. Add your preferred basemap
m.add_basemap("OpenTopoMap")
m.add_basemap("SATELLITE")
# 3. Stream your hosted COG
# cog_url = "https://leamhowe.github.io/data/SP_CNP_AllYears_pooled_2016-2026_COG.tif"
# (If you moved it to Hugging Face, paste that raw URL here instead)
cog_url = "https://huggingface.co/datasets/leamhowe/cairngorms-S2-snow-persistence/resolve/main/SP_CNP_AllYears_pooled_2016-2026_COG.tif"

sp = m.add_cog_layer(
    url=cog_url, 
    name="Pooled Data (2016-2026)",
    # colormap="viridis", 
    # vmin=0, vmax=75
    
    # rescale=f"{VMIN},{VMAX}",
    
    # colormap_name="viridis", vmin=0, vmax=75
)




# 4. Render
m

In [4]:
import folium
import httpx

# 1. Your file URL
# cog_url = "https://leamhowe.github.io/data/SP_CNP_AllYears_pooled_2016-2026_COG.tif"
cog_url = "https://huggingface.co/datasets/leamhowe/cairngorms-S2-snow-persistence/resolve/main/SP_CNP_AllYears_pooled_2016-2026_COG.tif"

# (Or your Hugging Face URL)

# 2. Ask TiTiler to generate a tile stream for your COG
titiler_endpoint = "https://api.cogeo.xyz" # Public TiTiler server
r = httpx.get(
    f"{titiler_endpoint}/cog/WebMercatorQuad/tilejson.json",
    params={"url": cog_url}
).json()

# 3. Get the bounding box so the map knows where to center
bounds = r["bounds"]
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

# 4. Initialize Folium Map
m = folium.Map(
    location=(center_lat, center_lon),
    zoom_start=10
)

# 5. Add OpenTopoMap
folium.TileLayer(
    tiles="https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png",
    attr="Map data: © OpenStreetMap contributors, SRTM | Map style: © OpenTopoMap (CC-BY-SA)",
    name="OpenTopoMap"
).add_to(m)

# 6. Add the TiTiler COG stream to Folium
folium.TileLayer(
    tiles=r["tiles"][0],
    attr="My COG Data",
    name="Pooled Data (2016-2026)",
    overlay=True
).add_to(m)

# Add a layer control panel to toggle layers on and off
folium.LayerControl().add_to(m)

# Render
m

ConnectError: [Errno 11001] getaddrinfo failed